# 04e_filter_2wikimultihop_for_reconstruction

Этот ноутбук читает результаты `03e_import_2wikimultihop.ipynb` и делает **более жёсткую фильтрацию** для поиска действительно полезных паттернов:

- режет `bridge_comparison` и shallow bridge-вопросы;
- понижает вес однотипных movie/director comparison-шаблонов;
- понижает вес слишком коротких `single-answer` цепочек;
- повышает вес `compositional` / `inference`;
- группирует по шаблонам вопроса, чтобы не тащить сотни почти одинаковых строк;
- собирает shortlist для ручного просмотра.

In [5]:
# =========================
# УСТАНОВКА ЗАВИСИМОСТЕЙ
# =========================
%pip install -q pandas pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
# =========================
# ИМПОРТЫ И КОНФИГ
# =========================
from __future__ import annotations

import csv
import json
import hashlib
import math
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

PROJECT_ROOT = Path.cwd()
INPUT_STAGE1 = PROJECT_ROOT / "artifacts_2wikimultihop_stage1"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_2wikimultihop_stage2"
JSONL_DIR = ARTIFACTS_DIR / "jsonl"
CSV_DIR = ARTIFACTS_DIR / "csv"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for p in [ARTIFACTS_DIR, JSONL_DIR, CSV_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

STRONG_STEM = "2wikimultihop_strong_multihop_candidates"
ALL_STEM = "2wikimultihop_all_normalized"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("INPUT_STAGE1 =", INPUT_STAGE1)
print("ARTIFACTS_DIR =", ARTIFACTS_DIR)


PROJECT_ROOT = /Users/matvey/Desktop/2wikimultihop
INPUT_STAGE1 = /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1
ARTIFACTS_DIR = /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage2


In [7]:
# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================
def normalize_spaces(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def read_text(path: Path) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def coerce_records(obj: Any) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for key in ["rows", "data", "items", "records"]:
            val = obj.get(key)
            if isinstance(val, list):
                return val
        return [obj]
    raise ValueError(f"Неожиданный тип данных: {type(obj)}")


def find_stage1_candidates(root: Path, stem: str) -> List[Path]:
    if not root.exists():
        return []
    found: List[Path] = []
    for p in root.rglob(f"{stem}.*"):
        if p.is_file() and p.suffix.lower() in {".json", ".jsonl"}:
            found.append(p)

    def sort_key(p: Path) -> Tuple[int, int, str]:
        # Предпочитаем .json, потом .jsonl; затем более короткий путь.
        ext_priority = 0 if p.suffix.lower() == ".json" else 1
        return (ext_priority, len(p.parts), str(p))

    return sorted(found, key=sort_key)


def read_concatenated_json_objects(text: str) -> List[Dict[str, Any]]:
    decoder = json.JSONDecoder()
    idx = 0
    n = len(text)
    rows: List[Dict[str, Any]] = []

    while idx < n:
        # Пропускаем whitespace, запятые и внешние скобки массива.
        while idx < n and text[idx] in ",[]":
            idx += 1
        if idx >= n:
            break

        obj, end = decoder.raw_decode(text, idx)
        rows.append(obj)
        idx = end

    return rows


def read_records(path: Path) -> List[Dict[str, Any]]:
    """
    Устойчивый ридер, который смотрит на содержимое, а не на расширение.
    Поддерживает:
    - обычный JSON array / JSON object
    - корректный JSONL
    - JSON array, сохранённый с расширением .jsonl
    - несколько JSON-объектов подряд: {..}
{..} или {..},{..}
    """
    text = read_text(path)
    stripped = text.lstrip("﻿").strip()
    if not stripped:
        return []

    # 1) Сначала пытаемся как обычный JSON целиком.
    try:
        obj = json.loads(stripped)
        return coerce_records(obj)
    except Exception:
        pass

    # 2) Затем как строгий JSONL (каждая строка — отдельный JSON).
    try:
        rows: List[Dict[str, Any]] = []
        for line in stripped.splitlines():
            line = line.strip().rstrip(",")
            if not line or line in {"[", "]"}:
                continue
            rows.append(json.loads(line))
        if rows:
            return rows
    except Exception:
        pass

    # 3) Затем как склеенные JSON-объекты / массив без доверия расширению.
    try:
        rows = read_concatenated_json_objects(stripped)
        if rows:
            return rows
    except Exception:
        pass

    preview = stripped[:500].replace("", " ")
    raise ValueError(
        f"Не удалось распарсить файл {path}. Первые символы: {preview!r}"
    )


def load_first_parseable(paths: List[Path]) -> Tuple[Path, List[Dict[str, Any]]]:
    errors = []
    for p in paths:
        try:
            rows = read_records(p)
            if rows:
                return p, rows
        except Exception as e:
            errors.append(f"{p}: {type(e).__name__}: {e}")

    msg = "\n".join(errors[:10])
    raise FileNotFoundError(
        "Не удалось прочитать ни один входной файл stage1. Попробованные варианты:" + msg
    )


def write_json(obj: Any, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def write_jsonl(rows: List[Dict[str, Any]], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


In [8]:
# =========================
# ЗАГРУЗКА ИСХОДНЫХ ДАННЫХ
# =========================
strong_candidates = find_stage1_candidates(INPUT_STAGE1, STRONG_STEM)
all_candidates = find_stage1_candidates(INPUT_STAGE1, ALL_STEM)

print("[strong candidate files]")
for p in strong_candidates:
    print(" -", p)
print("[all normalized files]")
for p in all_candidates:
    print(" -", p)

if strong_candidates:
    input_path, rows = load_first_parseable(strong_candidates)
    print(f"[load] using strong candidates file: {input_path}")
elif all_candidates:
    input_path, rows = load_first_parseable(all_candidates)
    print(f"[load] fallback to all normalized file: {input_path}")
else:
    raise FileNotFoundError(
        "Не найден input stage1. Сначала прогоните 03e_import_2wikimultihop.ipynb"
    )

df = pd.DataFrame(rows)
print("shape =", df.shape)
print("columns =", sorted(df.columns.tolist()))


[strong candidate files]
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_strong_multihop_candidates.json
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_strong_multihop_candidates.jsonl
[all normalized files]
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_all_normalized.json
 - /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_all_normalized.jsonl
[load] using strong candidates file: /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage1/jsonl/2wikimultihop_strong_multihop_candidates.json
shape = (1579, 26)
columns = ['answer_mode', 'benchmark_id', 'bridge_depth_estimate', 'context_docs', 'evidences', 'gold_answers', 'gold_qids', 'needs_rephrase', 'needs_translation', 'num_answers', 'num_context_docs', 'num_evidences', 'num_supporting_facts', 'question_en_original', 'question_ru', 'question_type', 'raw_answer', '

In [9]:
# =========================
# ВСПОМОГАТЕЛЬНЫЕ ПРИЗНАКИ ВОПРОСА
# =========================
def make_surface_template(question: str) -> str:
    q = normalize_spaces(question)
    q = q.lower()
    q = re.sub(r'"[^"]+"', "<quote>", q)
    q = re.sub(r"\b\d{4}\b", "<year>", q)
    q = re.sub(r"\b\d+\b", "<num>", q)

    # упрощённые entity-like спаны после частых конструкций
    q = re.sub(r"\b(the|a|an)\s+[a-z][a-z0-9'\- ]{2,40}\b", "<ent>", q)
    q = re.sub(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,4}\b", "<ent>", q)

    q = re.sub(r"\s+", " ", q).strip()

    # дополнительные нормализации типовых relation chains
    q = re.sub(r"which (film|movie) has the director who", "which <work> has the creator who", q)
    q = re.sub(r"which (film|movie) has the screenwriter who", "which <work> has the creator who", q)
    q = re.sub(r"where was the .+ born\??$", "where was <ent> born?", q)
    q = re.sub(r"who was the .+ of .+\??$", "who was the <role> of <ent>?", q)
    return q

def shallow_bridge_reason(question: str) -> str:
    q = normalize_spaces(question).lower()

    checks = [
        (r"^where was .+ born\??$", "where_born"),
        (r"^where is .+ from\??$", "where_from"),
        (r"^who (wrote|directed|produced|created|composed) .+\??$", "single_creator_relation"),
        (r"^who was the .+ of .+\??$", "single_role_relation"),
        (r"^what (platforms?|genre|nationality|instrument|city|country) .+\??$", "simple_attribute_lookup"),
        (r"^where did .+ go to university\??$", "education_lookup"),
        (r"^where was the (drummer|singer|actor|director|coach|quarterback).+ born\??$", "person_role_birthplace"),
    ]
    for pat, label in checks:
        if re.search(pat, q):
            return label

    if "director who was born" in q or "director who died" in q:
        return "movie_director_comparison"
    if "film has the director who" in q or "movie has the director who" in q:
        return "movie_director_comparison"
    if "which city did the band" in q:
        return "band_origin_lookup"
    if "what platforms can" in q:
        return "platform_lookup"

    return ""

def compute_domain_hint(question: str) -> str:
    q = normalize_spaces(question).lower()
    hints = [
        ("movies", ["film", "movie", "director", "screenplay", "actor", "actress"]),
        ("music", ["band", "singer", "drummer", "album", "song", "grammy"]),
        ("sports", ["world series", "quarterback", "coach", "fifa", "cup", "mvp", "team"]),
        ("politics", ["president", "senator", "state added to the union", "prime minister"]),
        ("videogames", ["video game", "animal crossing", "mass effect", "kingdom hearts", "blizzard"]),
        ("history", ["continent", "country", "war", "union"]),
        ("books", ["book", "novel", "literature", "author"]),
    ]
    for domain, needles in hints:
        if any(n in q for n in needles):
            return domain
    return "other"

def compute_question_family(question: str) -> str:
    q = normalize_spaces(question).lower()
    if q.startswith("which ") or q.startswith("what ") or q.startswith("who ") or q.startswith("where "):
        return "wh"
    return "other"

In [10]:
# =========================
# РЕКОНСТРУКТИВНАЯ ОЦЕНКА
# =========================
# Если stage1 брал уже candidate-пул, тут всё равно полезно пересчитать score более жёстко.

for col, default in [
    ("question_en_original", ""),
    ("question_type", ""),
    ("reasoning_family", ""),
    ("answer_mode", "unknown"),
    ("num_answers", 0),
    ("num_evidences", 0),
    ("num_supporting_facts", 0),
    ("bridge_depth_estimate", 0),
    ("shallow_bridge_flag", False),
]:
    if col not in df.columns:
        df[col] = default

df["question_en_original"] = df["question_en_original"].fillna("").astype(str)
df["question_type"] = df["question_type"].fillna("").astype(str)
df["reasoning_family"] = df["reasoning_family"].fillna(df["question_type"]).astype(str)
df["answer_mode"] = df["answer_mode"].fillna("unknown").astype(str)
for c in ["num_answers", "num_evidences", "num_supporting_facts", "bridge_depth_estimate"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

df["surface_template"] = df["question_en_original"].map(make_surface_template)
df["domain_hint"] = df["question_en_original"].map(compute_domain_hint)
df["question_family"] = df["question_en_original"].map(compute_question_family)
df["shallow_bridge_reason"] = df["question_en_original"].map(shallow_bridge_reason)
df["is_shallow_bridge_v2"] = df["shallow_bridge_reason"].ne("")
df["is_bridge_comparison"] = df["question_type"].eq("bridge_comparison")
df["is_compositional"] = df["question_type"].eq("compositional")
df["is_inference"] = df["question_type"].eq("inference")
df["is_comparison"] = df["question_type"].eq("comparison")
df["is_single_answer"] = df["answer_mode"].eq("single")
df["is_list_like"] = df["answer_mode"].eq("list_like")
df["has_multi_answers"] = df["num_answers"] >= 2
df["has_rich_evidence"] = (df["num_evidences"] >= 3) | (df["num_supporting_facts"] >= 3)
df["movie_director_pattern"] = df["shallow_bridge_reason"].eq("movie_director_comparison")

def score_row(r: pd.Series) -> int:
    score = 0

    # базовая ценность типа вопроса
    if r["is_compositional"]:
        score += 8
    if r["is_inference"]:
        score += 7
    if r["is_comparison"]:
        score += 4
    if r["is_bridge_comparison"]:
        score += 1  # но дальше дадим штрафы за shallow

    # evidence / support
    score += min(int(r["num_evidences"]), 6) * 2
    score += min(int(r["num_supporting_facts"]), 6)
    score += min(int(r["bridge_depth_estimate"]), 4)

    # answer richness
    if r["is_list_like"]:
        score += 5
    if r["has_multi_answers"]:
        score += 4
    if r["is_single_answer"]:
        score -= 2

    # penalties
    if bool(r["is_shallow_bridge_v2"]):
        score -= 8
    if bool(r["movie_director_pattern"]):
        score -= 6
    if bool(r["is_bridge_comparison"]) and bool(r["is_single_answer"]):
        score -= 3
    if int(r["num_evidences"]) <= 1 and int(r["num_supporting_facts"]) <= 2:
        score -= 2

    # positive nudges
    if bool(r["has_rich_evidence"]):
        score += 3
    if r["domain_hint"] not in {"other"}:
        score += 1

    return int(score)

df["reconstructability_score_v2"] = df.apply(score_row, axis=1)

def assign_bucket(r: pd.Series) -> str:
    s = int(r["reconstructability_score_v2"])
    if s >= 16:
        return "keep_for_reconstruction"
    if s >= 10:
        return "pattern_only"
    return "drop"

df["reconstruction_bucket_v2"] = df.apply(assign_bucket, axis=1)

In [11]:
# =========================
# СТАТИСТИКА ПОСЛЕ ЖЁСТКОЙ ФИЛЬТРАЦИИ
# =========================
summary = {
    "all_input_rows": int(len(df)),
    "by_question_type": df["question_type"].value_counts(dropna=False).to_dict(),
    "by_answer_mode": df["answer_mode"].value_counts(dropna=False).to_dict(),
    "by_bucket_v2": df["reconstruction_bucket_v2"].value_counts(dropna=False).to_dict(),
    "num_shallow_bridge_v2_true": int(df["is_shallow_bridge_v2"].sum()),
    "num_movie_director_pattern": int(df["movie_director_pattern"].sum()),
}
summary

{'all_input_rows': 1579,
 'by_question_type': {'bridge_comparison': 1533, 'comparison': 46},
 'by_answer_mode': {'single': 1579},
 'by_bucket_v2': {'drop': 1579},
 'num_shallow_bridge_v2_true': 1180,
 'num_movie_director_pattern': 1180}

In [12]:
# =========================
# СУЖЕНИЕ ДО БОЛЕЕ ПОЛЕЗНЫХ КАНДИДАТОВ
# =========================
filtered_df = df.copy()

# 1) режем совсем слабые bucket'и
filtered_df = filtered_df[filtered_df["reconstruction_bucket_v2"].isin(["keep_for_reconstruction", "pattern_only"])].copy()

# 2) в bucket pattern_only оставляем лишь часть, если шаблон не shallow и не movie/director
filtered_df = filtered_df[
    (filtered_df["reconstruction_bucket_v2"] == "keep_for_reconstruction")
    | (
        (filtered_df["reconstruction_bucket_v2"] == "pattern_only")
        & (~filtered_df["is_shallow_bridge_v2"])
        & (~filtered_df["movie_director_pattern"])
    )
].copy()

# 3) жёстко режем однотипные movie/director comparison вопросы
filtered_df = filtered_df[~filtered_df["movie_director_pattern"]].copy()

# 4) bridge_comparison оставляем только если есть богатый evidence и не single-answer shallow
filtered_df = filtered_df[
    (~filtered_df["is_bridge_comparison"])
    | (
        filtered_df["has_rich_evidence"]
        & (~filtered_df["is_shallow_bridge_v2"])
        & (
            filtered_df["is_list_like"]
            | filtered_df["has_multi_answers"]
            | filtered_df["is_compositional"]
            | filtered_df["is_inference"]
        )
    )
].copy()

# 5) убираем совсем короткие single-answer шаблоны
filtered_df = filtered_df[
    ~(
        filtered_df["is_single_answer"]
        & (filtered_df["num_evidences"] <= 1)
        & (filtered_df["num_supporting_facts"] <= 2)
        & filtered_df["is_shallow_bridge_v2"]
    )
].copy()

# 6) если перегнули с фильтрами — аккуратный fallback
if filtered_df.empty:
    print("[warn] aggressive filters produced 0 rows; relaxing selection...")
    filtered_df = df[
        (df["reconstruction_bucket_v2"] != "drop")
        & (~df["movie_director_pattern"])
    ].copy()

if filtered_df.empty:
    print("[warn] relaxed selection still produced 0 rows; using top-scored non-movie/director rows...")
    filtered_df = df[~df["movie_director_pattern"]].copy()
    filtered_df = filtered_df.sort_values(
        ["reconstructability_score_v2", "num_evidences", "num_supporting_facts", "num_answers"],
        ascending=[False, False, False, False]
    ).head(2000).copy()

filtered_df = filtered_df.sort_values(
    [
        "reconstructability_score_v2",
        "num_evidences",
        "num_supporting_facts",
        "num_answers",
    ],
    ascending=[False, False, False, False]
).reset_index(drop=True)

print("filtered_df.shape =", filtered_df.shape)
print("filtered by question_type =", filtered_df["question_type"].value_counts(dropna=False).to_dict())
filtered_df[[
    "question_en_original", "question_type", "answer_mode",
    "num_answers", "num_evidences", "num_supporting_facts",
    "reconstructability_score_v2", "reconstruction_bucket_v2"
]].head(10)


[warn] aggressive filters produced 0 rows; relaxing selection...
[warn] relaxed selection still produced 0 rows; using top-scored non-movie/director rows...
filtered_df.shape = (399, 42)
filtered by question_type = {'bridge_comparison': 353, 'comparison': 46}


,question_en_original,question_type,answer_mode,num_answers,num_evidences,num_supporting_facts,reconstructability_score_v2,reconstruction_bucket_v2
0,"Which film came out first, Demons Of The Mind ...",comparison,single,1,0,0,5,drop
1,"Which film came out first, Jonah Who Lived In ...",comparison,single,1,0,0,4,drop
2,"Which film was released first, The Crew Of The...",comparison,single,1,0,0,4,drop
3,"Which film came out earlier, Cattle Drive or P...",comparison,single,1,0,0,4,drop
4,"Which film came out earlier, Tea In The Harem ...",comparison,single,1,0,0,4,drop
5,Are both Wizards of the Lost Kingdom II and Wh...,comparison,single,1,0,0,4,drop
6,"Which film was released earlier, Desire In The...",comparison,single,1,0,0,4,drop
7,"Which film was released more recently, Lost In...",comparison,single,1,0,0,4,drop
8,"Which film was released more recently, House N...",comparison,single,1,0,0,4,drop
9,"Which film was released more recently, House O...",comparison,single,1,0,0,4,drop


In [13]:
# =========================
# BANK OF PATTERNS: ГРУППИРОВКА ПО ШАБЛОНАМ
# =========================
# Хотим не сотни почти одинаковых строк, а банк более разнообразных паттернов.
group_cols = ["surface_template", "question_type", "answer_mode"]

pattern_rows = []
for keys, group in filtered_df.groupby(group_cols, dropna=False):
    surface_template, q_type, answer_mode = keys
    group = group.sort_values(
        ["reconstructability_score_v2", "num_evidences", "num_supporting_facts", "num_answers"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    rep = group.iloc[0].to_dict()
    rep["pattern_count"] = int(len(group))
    rep["pattern_score_max"] = float(group["reconstructability_score_v2"].max())
    rep["pattern_score_mean"] = float(group["reconstructability_score_v2"].mean())
    rep["pattern_domains"] = sorted(set(group["domain_hint"].astype(str).tolist()))
    pattern_rows.append(rep)

if pattern_rows:
    pattern_df = pd.DataFrame(pattern_rows).sort_values(
        ["pattern_score_max", "pattern_count", "pattern_score_mean"],
        ascending=[False, False, False]
    ).reset_index(drop=True)
else:
    print("[warn] pattern_rows is empty; creating empty pattern_df")
    pattern_df = pd.DataFrame(columns=list(filtered_df.columns) + [
        "pattern_count", "pattern_score_max", "pattern_score_mean", "pattern_domains"
    ])

print("pattern_df.shape =", pattern_df.shape)
if not pattern_df.empty:
    pattern_df[[
        "surface_template", "question_type", "answer_mode",
        "pattern_count", "pattern_score_max", "pattern_domains"
    ]].head(20)
else:
    pattern_df.head()


pattern_df.shape = (366, 46)


In [14]:
# =========================
# SHORTLIST ДЛЯ РУЧНОГО ПРОСМОТРА
# =========================
# Берём ограниченное число паттернов на тип, чтобы shortlist был разнообразным.
max_per_type = {
    "compositional": 40,
    "inference": 35,
    "comparison": 25,
    "bridge_comparison": 10,
}

short_rows = []
used_templates = set()
type_counts = Counter()

for _, row in pattern_df.iterrows():
    q_type = str(row.get("question_type", ""))
    tmpl = str(row.get("surface_template", ""))
    if tmpl in used_templates:
        continue

    limit = max_per_type.get(q_type, 10)
    if type_counts[q_type] >= limit:
        continue

    short_rows.append(row.to_dict())
    used_templates.add(tmpl)
    type_counts[q_type] += 1

# fallback: если shortlist пустой, берём top-N из pattern_df
if not short_rows and not pattern_df.empty:
    print("[warn] shortlist from per-type limits is empty; taking top rows directly")
    short_rows = pattern_df.head(80).to_dict(orient="records")

shortlist_df = pd.DataFrame(short_rows)
if not shortlist_df.empty:
    shortlist_df = shortlist_df.sort_values(
        ["reconstructability_score_v2", "pattern_count", "num_evidences", "num_supporting_facts"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

print("shortlist_df.shape =", shortlist_df.shape)
print("shortlist by question_type =", shortlist_df["question_type"].value_counts(dropna=False).to_dict() if not shortlist_df.empty else {})
if not shortlist_df.empty:
    shortlist_df[[
        "question_en_original", "question_type", "answer_mode",
        "num_answers", "pattern_count", "reconstructability_score_v2"
    ]].head(20)
else:
    shortlist_df.head()


shortlist_df.shape = (35, 46)
shortlist by question_type = {'comparison': 25, 'bridge_comparison': 10}


In [15]:
# =========================
# MANUAL REVIEW SHEET
# =========================
manual_review_df = shortlist_df[[
    "benchmark_id",
    "question_en_original",
    "surface_template",
    "question_type",
    "reasoning_family",
    "answer_mode",
    "num_answers",
    "num_evidences",
    "num_supporting_facts",
    "bridge_depth_estimate",
    "domain_hint",
    "reconstructability_score_v2",
    "reconstruction_bucket_v2",
    "pattern_count",
    "pattern_score_max",
    "pattern_score_mean",
    "pattern_domains",
]].copy()

manual_review_df["manual_decision"] = ""
manual_review_df["manual_bucket"] = ""
manual_review_df["notes"] = ""

print("manual_review_df.shape =", manual_review_df.shape)
manual_review_df.head(10)

manual_review_df.shape = (35, 20)


,benchmark_id,question_en_original,surface_template,question_type,reasoning_family,answer_mode,num_answers,num_evidences,num_supporting_facts,bridge_depth_estimate,domain_hint,reconstructability_score_v2,reconstruction_bucket_v2,pattern_count,pattern_score_max,pattern_score_mean,pattern_domains,manual_decision,manual_bucket,notes
0,2wiki_d8e5f5d6087011ebbd63ac1f6bf848b6,"Which film came out first, Demons Of The Mind ...","which film came out first, demons of <ent>?",comparison,comparison,single,1,0,0,4,movies,5,drop,1,5.0,5.0,[movies],,,
1,2wiki_e8c28d90085511ebbd59ac1f6bf848b6,Are both Cathedral Basilica Of The Nativity Of...,"are both cathedral basilica of <ent>, tarnów a...",comparison,comparison,single,1,0,0,3,history,4,drop,1,4.0,4.0,[history],,,
2,2wiki_a95db437085511ebbd59ac1f6bf848b6,"Are both churches, Cathedral Of The Assumption...","are both churches, cathedral of <ent>, rožňava...",comparison,comparison,single,1,0,0,3,history,4,drop,1,4.0,4.0,[history],,,
3,2wiki_d4a9df0e08ff11ebbdadac1f6bf848b6,Are both Wizards of the Lost Kingdom II and Wh...,are both wizards of <ent> osama bin laden? fro...,comparison,comparison,single,1,0,0,3,history,4,drop,1,4.0,4.0,[history],,,
4,2wiki_75f12b40085611ebbd59ac1f6bf848b6,Are Church Of The Multiplication and Church Of...,"are church of <ent> (pulaski, tennessee) locat...",comparison,comparison,single,1,0,0,3,history,4,drop,1,4.0,4.0,[history],,,
5,2wiki_6e74571a085711ebbd5aac1f6bf848b6,Are Iglesia De Santa María (Arbazal) and Churc...,are iglesia de santa maría (arbazal) and churc...,comparison,comparison,single,1,0,0,3,history,4,drop,1,4.0,4.0,[history],,,
6,2wiki_d4ab11de08b211ebbd85ac1f6bf848b6,Do the movies In The Land Of The Head Hunters ...,"do <ent>and out of <ent>, originate from <ent>?",comparison,comparison,single,1,0,0,3,movies,4,drop,1,4.0,4.0,[movies],,,
7,2wiki_62b39e73086611ebbd5eac1f6bf848b6,"Which film came out earlier, The Notorious Mrs...","which film came out earlier, <ent>. carrick or...",comparison,comparison,single,1,0,0,3,movies,4,drop,1,4.0,4.0,[movies],,,
8,2wiki_272e867e091d11ebbdaeac1f6bf848b6,"Which film came out earlier, The Flight In The...","which film came out earlier, <ent>?",comparison,comparison,single,1,0,0,3,movies,4,drop,1,4.0,4.0,[movies],,,
9,2wiki_d086294a095511ebbdaeac1f6bf848b6,"Which film came out earlier, Battle Of The Cor...","which film came out earlier, battle of <ent> (...",comparison,comparison,single,1,0,0,3,movies,4,drop,1,4.0,4.0,[movies],,,


In [16]:
# =========================
# TRANSLATION MINIMAL
# =========================
translation_minimal_df = shortlist_df[[
    "benchmark_id",
    "source_dataset",
    "source_split",
    "question_en_original",
    "question_type",
    "reasoning_family",
    "answer_mode",
    "num_answers",
    "num_evidences",
    "num_supporting_facts",
    "reconstructability_score_v2",
]].copy()

translation_minimal_df["translation_ready_text"] = translation_minimal_df["question_en_original"].astype(str)
translation_minimal_df["needs_translation"] = True
translation_minimal_df["needs_rephrase"] = True

print("translation_minimal_df.shape =", translation_minimal_df.shape)

translation_minimal_df.shape = (35, 14)


In [17]:
# =========================
# ЭКСПОРТ
# =========================
filtered_records = filtered_df.to_dict(orient="records")
pattern_records = pattern_df.to_dict(orient="records")
shortlist_records = shortlist_df.to_dict(orient="records")
manual_review_records = manual_review_df.to_dict(orient="records")
translation_minimal_records = translation_minimal_df.to_dict(orient="records")

write_jsonl(filtered_records, JSONL_DIR / "2wikimultihop_filtered_candidates.jsonl")
write_json(filtered_records, JSONL_DIR / "2wikimultihop_filtered_candidates.json")
filtered_df.to_csv(CSV_DIR / "2wikimultihop_filtered_candidates.csv", index=False)

write_jsonl(pattern_records, JSONL_DIR / "2wikimultihop_pattern_bank.jsonl")
write_json(pattern_records, JSONL_DIR / "2wikimultihop_pattern_bank.json")
pattern_df.to_csv(CSV_DIR / "2wikimultihop_pattern_bank.csv", index=False)

write_jsonl(shortlist_records, JSONL_DIR / "2wikimultihop_reconstruction_shortlist.jsonl")
write_json(shortlist_records, JSONL_DIR / "2wikimultihop_reconstruction_shortlist.json")
shortlist_df.to_csv(CSV_DIR / "2wikimultihop_reconstruction_shortlist.csv", index=False)

write_jsonl(translation_minimal_records, JSONL_DIR / "2wikimultihop_translation_minimal_v2.jsonl")
write_json(translation_minimal_records, JSONL_DIR / "2wikimultihop_translation_minimal_v2.json")
translation_minimal_df.to_csv(CSV_DIR / "2wikimultihop_translation_minimal_v2.csv", index=False)

manual_review_df.to_csv(CSV_DIR / "2wikimultihop_manual_review_sheet.csv", index=False)

summary_v2 = {
    "filtered_candidates": int(len(filtered_df)),
    "pattern_bank": int(len(pattern_df)),
    "reconstruction_shortlist": int(len(shortlist_df)),
    "manual_review_sheet": int(len(manual_review_df)),
    "translation_minimal": int(len(translation_minimal_df)),
    "shortlist_by_type": shortlist_df["question_type"].value_counts(dropna=False).to_dict() if not shortlist_df.empty else {},
    "shortlist_by_answer_mode": shortlist_df["answer_mode"].value_counts(dropna=False).to_dict() if not shortlist_df.empty else {},
}
write_json(summary_v2, REPORTS_DIR / "summary_stage2.json")

print("[saved] stage2 artifacts written to", ARTIFACTS_DIR)
summary_v2

[saved] stage2 artifacts written to /Users/matvey/Desktop/2wikimultihop/artifacts_2wikimultihop_stage2


{'filtered_candidates': 399,
 'pattern_bank': 366,
 'reconstruction_shortlist': 35,
 'manual_review_sheet': 35,
 'translation_minimal': 35,
 'shortlist_by_type': {'comparison': 25, 'bridge_comparison': 10},
 'shortlist_by_answer_mode': {'single': 35}}

In [18]:
# =========================
# PREVIEW
# =========================
display_cols = [
    "benchmark_id",
    "question_en_original",
    "surface_template",
    "question_type",
    "answer_mode",
    "num_answers",
    "num_evidences",
    "num_supporting_facts",
    "reconstructability_score_v2",
    "pattern_count",
]
shortlist_df[display_cols].head(30)

,benchmark_id,question_en_original,surface_template,question_type,answer_mode,num_answers,num_evidences,num_supporting_facts,reconstructability_score_v2,pattern_count
0,2wiki_d8e5f5d6087011ebbd63ac1f6bf848b6,"Which film came out first, Demons Of The Mind ...","which film came out first, demons of <ent>?",comparison,single,1,0,0,5,1
1,2wiki_e8c28d90085511ebbd59ac1f6bf848b6,Are both Cathedral Basilica Of The Nativity Of...,"are both cathedral basilica of <ent>, tarnów a...",comparison,single,1,0,0,4,1
2,2wiki_a95db437085511ebbd59ac1f6bf848b6,"Are both churches, Cathedral Of The Assumption...","are both churches, cathedral of <ent>, rožňava...",comparison,single,1,0,0,4,1
3,2wiki_d4a9df0e08ff11ebbdadac1f6bf848b6,Are both Wizards of the Lost Kingdom II and Wh...,are both wizards of <ent> osama bin laden? fro...,comparison,single,1,0,0,4,1
4,2wiki_75f12b40085611ebbd59ac1f6bf848b6,Are Church Of The Multiplication and Church Of...,"are church of <ent> (pulaski, tennessee) locat...",comparison,single,1,0,0,4,1
5,2wiki_6e74571a085711ebbd5aac1f6bf848b6,Are Iglesia De Santa María (Arbazal) and Churc...,are iglesia de santa maría (arbazal) and churc...,comparison,single,1,0,0,4,1
6,2wiki_d4ab11de08b211ebbd85ac1f6bf848b6,Do the movies In The Land Of The Head Hunters ...,"do <ent>and out of <ent>, originate from <ent>?",comparison,single,1,0,0,4,1
7,2wiki_62b39e73086611ebbd5eac1f6bf848b6,"Which film came out earlier, The Notorious Mrs...","which film came out earlier, <ent>. carrick or...",comparison,single,1,0,0,4,1
8,2wiki_272e867e091d11ebbdaeac1f6bf848b6,"Which film came out earlier, The Flight In The...","which film came out earlier, <ent>?",comparison,single,1,0,0,4,1
9,2wiki_d086294a095511ebbdaeac1f6bf848b6,"Which film came out earlier, Battle Of The Cor...","which film came out earlier, battle of <ent> (...",comparison,single,1,0,0,4,1
